# Demo guiada — estados y confianza en una interfaz de IA

La demo usa el controlador de la solución docente y un gateway determinista. Recorre el mismo contrato de salida que S3/S4 sin necesitar un `.joblib`.

La confianza es una señal del clasificador, no una garantía de calidad ni una recomendación profesional.

In [ ]:
from pathlib import Path
import sys

repo_root = next(parent for parent in (Path.cwd(), *Path.cwd().parents)
                 if (parent / 'semana6/modules/06-ux-model-consumption').is_dir())
solution_src = repo_root / 'semana6/modules/06-ux-model-consumption/solutions/02-robust-streamlit/src'
if str(solution_src) not in sys.path:
    sys.path.insert(0, str(solution_src))

from model_ui.controller import PredictionController
from model_ui.errors import InputContractError
from model_ui.gateway import DemoGateway
from model_ui.telemetry import Telemetry

SAMPLE = {
    'fixed_acidity': 7.4, 'volatile_acidity': 0.7, 'citric_acid': 0.0,
    'residual_sugar': 1.9, 'chlorides': 0.076,
    'free_sulfur_dioxide': 11.0, 'total_sulfur_dioxide': 34.0,
    'density': 0.9978, 'ph': 3.51, 'sulphates': 0.56, 'alcohol': 9.4,
}

## 1. Éxito con confianza media

Observa que la secuencia tiene un estado `loading` antes del resultado final.

In [ ]:
events = []
telemetry = Telemetry()
controller = PredictionController(DemoGateway(confidence=0.74), telemetry=telemetry)
state = controller.submit(SAMPLE, emit=events.append)
print('transiciones:', [event.phase for event in events])
print('resultado:', state.view.quality_label)
print('confianza:', state.view.confidence_label)
print('latencia:', state.view.latency_label)
print('telemetría:', telemetry.snapshot())

## 2. Confianza baja

La UI debe recomendar revisión sin ocultar el resultado ni llamarlo seguro.

In [ ]:
low_state = PredictionController(
    DemoGateway(confidence=0.42), telemetry=telemetry
).submit(SAMPLE)
print(low_state.view.confidence_level)
print(low_state.view.confidence_message)

## 3. Error de contrato

El usuario recibe una acción de recuperación y un identificador; no recibe el traceback.

In [ ]:
class BrokenGateway:
    def predict(self, values):
        raise InputContractError('ph fuera de rango')

error_state = PredictionController(BrokenGateway(), telemetry=telemetry).submit(SAMPLE)
print(error_state.phase)
print(error_state.error.code, error_state.error.title)
print(error_state.error.recovery)
print('telemetría:', telemetry.snapshot())

## Preguntas para el debrief

- ¿Qué diferencia hay entre `confidence_level` y una decisión de negocio?
- ¿Qué información de S4 permite investigar una respuesta?
- ¿Qué cambiaría si el gateway llamase a HTTP en la semana 7?
- ¿Qué dato del formulario no aparece en `TelemetrySnapshot` y por qué?